# A100 행렬곱 벤치마크와 프로파일링

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/infer-opt/blob/main/notebooks/a100_matmul_profiling.ipynb)

Colab에서 **런타임 → 런타임 유형 변경 → A100 GPU**를 선택하고 위에서부터 실행합니다. 별도 모델·저장소 clone·패키지 설치가 필요 없습니다.

워크로드: `C[M,N] = A[M,K] @ B[K,N]`, contiguous 입력, `torch.mm(..., out=C)`, bias/activation 없음. FP32(strict), FP32(TF32 허용), FP16, BF16을 비교합니다. TF32 허용은 실제 Tensor Core 사용을 보장하지 않으며 kernel trace로 확인합니다.

기본 shape는 decode형 M=1/16과 prefill형 M=256/2048, 정사각 GEMM입니다. **모델 전체 성능이 아닌 동일 입력을 반복하는 warmed GEMM 측정**입니다. 입력 생성·출력 할당·정확도 검사·profiler 비용은 타이밍에서 제외합니다.

In [ ]:
import csv, json, platform, shutil, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
assert 'A100' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
torch.cuda.set_device(0)
torch.manual_seed(42)
OUT = Path('a100_matmul_profile')
OUT.mkdir(exist_ok=True)
metadata = {
    'gpu': torch.cuda.get_device_name(0),
    'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
    'compute_capability': list(torch.cuda.get_device_capability(0)),
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'python': platform.python_version(),
}
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
(OUT / 'nvidia-smi.txt').write_text(r.stdout + r.stderr)
print(r.stdout)
print(json.dumps(metadata, indent=2))

## 설정
`SHAPES`의 순서는 `(M, K, N)`입니다. 실제 linear layer `X @ W.T`를 재현하려면 weight stride도 달라질 수 있으므로 이 노트북의 contiguous B 조건과 구분하세요.

CUDA event 한 쌍으로 `INNER`회 호출을 재고 호출당 시간으로 나눕니다. 표의 p50/p95는 **단일 호출 tail latency가 아니라 반복 묶음의 호출당 평균 시간 분포**입니다. 작은 GEMM은 Python dispatch와 kernel 사이 빈 시간이 포함될 수 있습니다.

In [ ]:
SHAPES = [(1, 4096, 4096), (16, 4096, 4096),
          (256, 4096, 4096), (2048, 4096, 4096), (4096, 4096, 4096)]
MODES = {'fp32_strict': torch.float32, 'tf32_allowed': torch.float32,
         'fp16': torch.float16, 'bf16': torch.bfloat16}
WARMUP, REPEATS, INNER = 20, 30, 20
PROFILE_SHAPE = (2048, 4096, 4096)
PROFILE_MODE = 'bf16'
PROFILE_STEPS = 5
# NMSE는 입력을 각 dtype으로 반올림한 뒤 계산한 FP32(strict) 참조값과의 차이입니다.
# 실험용 통과 기준이며, 임의의 입력에 대한 품질을 보장하지 않습니다.
NMSE_LIMIT = {'fp32_strict': 1e-10, 'tf32_allowed': 1e-5,
              'fp16': 1e-5, 'bf16': 1e-4}

def set_mode(mode):
    torch.set_float32_matmul_precision('high' if mode == 'tf32_allowed' else 'highest')
    torch.backends.cuda.matmul.allow_tf32 = mode == 'tf32_allowed'
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

metadata['config'] = dict(shapes=SHAPES, modes=list(MODES), warmup=WARMUP,
                          repeats=REPEATS, inner=INNER, seed=42,
                          profile_shape=PROFILE_SHAPE, profile_mode=PROFILE_MODE,
                          nmse_limits=NMSE_LIMIT, reduced_precision_reduction=False)
(OUT / 'metadata.json').write_text(json.dumps(metadata, indent=2))

In [ ]:
@torch.inference_mode()
def benchmark_shape(shape):
    m, k, n = shape
    # 모든 mode에서 원본 FP32 입력 공유
    a0 = torch.randn(m, k, device='cuda')
    b0 = torch.randn(k, n, device='cuda')
    results, raw = [], []
    for mode, dtype in MODES.items():
        a, b = a0.to(dtype), b0.to(dtype)
        c = torch.empty(m, n, device='cuda', dtype=dtype)
        set_mode('fp32_strict')
        ref = torch.mm(a.float(), b.float())
        set_mode(mode)
        torch.mm(a, b, out=c)
        delta = c.float() - ref
        nmse = (delta.square().sum() / ref.square().sum().clamp_min(1e-30)).item()
        max_abs = delta.abs().max().item()
        assert torch.isfinite(c).all().item() and nmse <= NMSE_LIMIT[mode], (mode, shape, nmse)
        del ref, delta
        for _ in range(WARMUP):
            torch.mm(a, b, out=c)
        torch.cuda.synchronize()
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        samples = []
        for trial in range(REPEATS):
            start.record()
            for _ in range(INNER):
                torch.mm(a, b, out=c)
            end.record()
            end.synchronize()
            ms = start.elapsed_time(end) / INNER
            samples.append(ms)
            raw.append(dict(M=m, K=k, N=n, mode=mode, trial=trial, ms=ms))
        p50, p95 = np.percentile(samples, [50, 95])
        # 입출력을 한 번씩 전송한다고 가정한 최소 bytes. 실측 DRAM bytes가 아님.
        min_bytes = (m*k + k*n + m*n) * a.element_size()
        results.append(dict(M=m, K=k, N=n, mode=mode, p50_ms=float(p50), p95_ms=float(p95),
                            TFLOPs=2*m*k*n/(p50*1e9),
                            ideal_arithmetic_intensity=2*m*k*n/min_bytes,
                            nmse=nmse, max_abs=max_abs))
        del a, b, c
    return results, raw

rows, raw = [], []
for shape in SHAPES:
    result, samples = benchmark_shape(shape)
    rows.extend(result)
    raw.extend(samples)
    print('finished:', shape)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'summary.csv', index=False)
pd.DataFrame(raw).to_csv(OUT / 'samples.csv', index=False)
display(df.round(6))

## CUDA kernel trace
타이밍 측정과 별도로 profiler를 실행합니다. 표의 CPU 시간은 dispatch, CUDA/device 시간은 GPU 실행 시간입니다. `trace.json`을 https://ui.perfetto.dev 에서 열어 CUDA kernel 이름과 실행 간격을 확인하세요.

이 trace는 kernel 시간과 실행 순서를 보여줍니다. **Tensor Core 이용률·SM occupancy·실측 DRAM 대역폭은 이 표로 알 수 없습니다.** 해당 하드웨어 카운터는 별도의 Nsight Compute와 profiling 권한이 필요합니다. 작은 M/큰 M으로 `PROFILE_SHAPE`를 바꿔 다시 실행해 비교하세요.

Profiler 공식 문서: https://docs.pytorch.org/docs/stable/profiler.html

In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function

assert PROFILE_MODE in MODES
m, k, n = PROFILE_SHAPE
dtype = MODES[PROFILE_MODE]
set_mode(PROFILE_MODE)
a = torch.randn(m, k, device='cuda', dtype=dtype)
b = torch.randn(k, n, device='cuda', dtype=dtype)
c = torch.empty(m, n, device='cuda', dtype=dtype)
with torch.inference_mode():
    for _ in range(WARMUP):
        torch.mm(a, b, out=c)
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                 record_shapes=True, profile_memory=True) as prof:
        for _ in range(PROFILE_STEPS):
            with record_function(f'gemm_{PROFILE_MODE}_{m}x{k}x{n}'):
                torch.mm(a, b, out=c)
        torch.cuda.synchronize()
prof.export_chrome_trace(str(OUT / 'trace.json'))
table = prof.key_averages(group_by_input_shape=True).table(
    sort_by='self_cuda_time_total', row_limit=25)
print(table)
(OUT / 'profiler.txt').write_text(table)
# CPU trace만 있는 경우 실패로 표시
cuda_events = [e for e in prof.events() if e.device_type == torch.autograd.DeviceType.CUDA]
print('CUDA events:', len(cuda_events))
assert cuda_events, 'CUDA trace가 없습니다. CUPTI/profiler 경고를 확인하세요.'
print('kernel names:', sorted({e.name for e in cuda_events}))
del a, b, c

## 결과 저장과 해석
- `summary.csv`: p50/p95, 유효 TFLOP/s (`2MKN / 시간`), 정확도 진단.
- `samples.csv`: 모든 반복 묶음의 호출당 평균 시간.
- `trace.json`, `profiler.txt`: GPU kernel과 CPU dispatch 분석.
- `metadata.json`, `nvidia-smi.txt`: 실제 GPU와 소프트웨어·측정 설정.

M=1은 작은 연산량과 weight 읽기/호출 비용, 큰 M은 연산 처리량이 영향을 줄 수 있습니다. 병목 판정에는 trace 및 추가 카운터가 필요합니다. TF32/저정밀 속도 차이만으로 Tensor Core 사용을 확정하지 마세요. 입력은 매번 동일하고 cache를 비우지 않으므로 결과에 cache 재사용이 반영됩니다.

아래 셀은 ZIP을 다운로드합니다. 완료 후 Colab 런타임을 연결 해제·삭제하세요.

## Nsight Compute로 실행한 커널의 SASS 확인 (선택)
기존 trace와 같은 `PROFILE_SHAPE`, `PROFILE_MODE`를 별도 프로세스에서 실행합니다. warmup 이후 `cudaProfilerStart/Stop` 구간의 GEMM 한 번만 수집합니다. PyTorch profiler와 동시에 실행하지 않습니다.

아래 셀을 실행하면 `nsight/gemm.ncu-rep`와 `nsight/sass.txt`가 마지막 ZIP에 포함됩니다. `.ncu-rep`를 로컬 **Nsight Compute GUI → 보고서의 커널 선택 → Source → SASS**에서 여세요. `sass.txt`로도 명령어를 확인할 수 있습니다. BF16 커널의 `HMMA` 계열 명령어를 찾아 Tensor Core 연산을 살펴보세요. 소스/PTX가 없는 배포 라이브러리에서도 SASS를 확인할 수 있습니다.

`RUN_NCU = True`로 바꾸고 실행하세요. ncu가 없으면 출력에 표시된 NVIDIA 공식 다운로드에서 Linux용 Nsight Compute를 설치한 뒤 다시 실행합니다. `ERR_NVGPUCTRPERM`은 GPU 성능 카운터 접근 권한 문제이며, Colab에서는 호스트 설정을 바꿀 수 없어 수집이 막힐 수 있습니다. 실패 로그도 보존합니다. NCU의 replay·cache·clock 설정이 일반 실행과 다를 수 있으므로 여기서 얻은 시간은 앞의 벤치마크와 직접 비교하지 않습니다.

[NVIDIA CLI 문서](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html) · [Nsight Compute 다운로드](https://developer.nvidia.com/tools-overview/nsight-compute/get-started)

In [ ]:
RUN_NCU = False  # SASS를 수집하려면 True로 변경

if RUN_NCU:
    import glob, re, sys, textwrap
    nsight_dir = OUT / 'nsight'
    nsight_dir.mkdir(exist_ok=True)
    candidates = [shutil.which('ncu')]
    candidates += sorted(glob.glob('/usr/local/cuda*/nsight-compute*/ncu'))
    candidates += sorted(glob.glob('/opt/nvidia/nsight-compute*/ncu'))
    ncu = next((str(Path(x).resolve()) for x in candidates if x and Path(x).is_file()), None)
    if ncu is None:
        raise RuntimeError('ncu가 없습니다. NVIDIA 공식 Linux용 Nsight Compute를 설치하고 다시 실행하세요: '
                           'https://developer.nvidia.com/tools-overview/nsight-compute/get-started')
    version = subprocess.run([ncu, '--version'], capture_output=True, text=True)
    (nsight_dir / 'ncu-version.txt').write_text(version.stdout + version.stderr)
    print(version.stdout + version.stderr)
    worker = nsight_dir / 'profile_gemm.py'
    worker.write_text(textwrap.dedent("""
        import json, sys, torch
        m, k, n = map(int, sys.argv[1:4])
        mode = sys.argv[4]
        assert torch.cuda.is_available()
        assert 'A100' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
        torch.manual_seed(42)
        dtype = {'fp32_strict': torch.float32, 'tf32_allowed': torch.float32,
                 'fp16': torch.float16, 'bf16': torch.bfloat16}[mode]
        torch.set_float32_matmul_precision('high' if mode == 'tf32_allowed' else 'highest')
        torch.backends.cuda.matmul.allow_tf32 = mode == 'tf32_allowed'
        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        a = torch.randn(m, k, device='cuda', dtype=dtype)
        b = torch.randn(k, n, device='cuda', dtype=dtype)
        c = torch.empty(m, n, device='cuda', dtype=dtype)
        with torch.inference_mode():
            for _ in range(20):
                torch.mm(a, b, out=c)
            torch.cuda.synchronize()
            print(json.dumps({'gpu': torch.cuda.get_device_name(0), 'shape': [m,k,n],
                              'mode': mode, 'torch': torch.__version__}), flush=True)
            torch.cuda.cudart().cudaProfilerStart()
            torch.mm(a, b, out=c)
            torch.cuda.synchronize()
            torch.cuda.cudart().cudaProfilerStop()
    """).strip() + '\n')
    report_base = nsight_dir / 'gemm'
    cmd = [ncu, '--target-processes', 'all', '--profile-from-start', 'off',
           '--set', 'basic', '--launch-count', '1', '--force-overwrite',
           '--export', str(report_base), sys.executable, str(worker),
           *map(str, PROFILE_SHAPE), PROFILE_MODE]
    (nsight_dir / 'command.json').write_text(json.dumps(cmd, indent=2))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    log = result.stdout + result.stderr
    (nsight_dir / 'capture.log').write_text(log)
    print(log)
    if result.returncode != 0 or not report_base.with_suffix('.ncu-rep').is_file():
        if 'ERR_NVGPUCTRPERM' in log:
            raise RuntimeError('Colab 호스트의 GPU counter 접근 권한이 없어 수집할 수 없습니다. capture.log를 확인하세요.')
        raise RuntimeError('NCU 수집 실패: nsight/capture.log를 확인하세요.')
    exported = subprocess.run([ncu, '--import', str(report_base.with_suffix('.ncu-rep')),
                               '--page', 'source', '--print-source', 'sass'],
                              capture_output=True, text=True, timeout=120)
    (nsight_dir / 'sass.txt').write_text(exported.stdout)
    (nsight_dir / 'sass-export.log').write_text(exported.stderr)
    if exported.returncode != 0:
        raise RuntimeError('SASS 내보내기 실패: sass-export.log를 확인하세요.')
    sass = exported.stdout
    if not re.search(r'\b(?:HMMA|MMA|LDG|MOV|S2R)\b', sass):
        raise RuntimeError('SASS 명령어를 찾지 못했습니다. 보고서를 GUI에서 열어 확인하세요.')
    print(sass[:16000])
    print('HMMA 포함 행 수:', sum('HMMA' in line for line in sass.splitlines()))
    print('GUI에서 열 파일:', report_base.with_suffix('.ncu-rep'))
else:
    print('NCU 수집 생략. RUN_NCU=True로 바꾸면 수집합니다.')

In [ ]:
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print(archive)
from google.colab import files
files.download(archive)